In [4]:
import cv2
import pandas as pd
from ultralytics import YOLO

# ---------------------------
# CONFIG
# ---------------------------
VIDEO_PATH = "videos/__WFqm4i3vE_00.mp4"#"sim_dataset/videos/head-on/Town03_head-on_clear_00.mp4"
OUTPUT_VIDEO_PATH = "output_tracked.mp4"
OUTPUT_CSV_PATH = "tracking_data.csv"

MODEL_NAME = "yolov9m.pt"  # try yolov8l.pt for better accuracy

# COCO class IDs for vehicles
VEHICLE_CLASSES = [2, 3, 5, 7]  
# 2=car, 3=motorcycle, 5=bus, 7=truck

# ---------------------------
# LOAD MODEL
# ---------------------------
model = YOLO(MODEL_NAME)

# ---------------------------
# VIDEO SETUP
# ---------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# ---------------------------
# DATA STORAGE
# ---------------------------
data = []

frame_idx = 0

# ---------------------------
# PROCESS VIDEO
# ---------------------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run tracking
    results = model.track(
        frame,
        persist=True,   # keeps track IDs across frames
        classes=VEHICLE_CLASSES,
        conf=0.3
    )

    if results[0].boxes is not None:
        boxes = results[0].boxes

        for box in boxes:
            # Bounding box
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            # Track ID
            track_id = int(box.id[0]) if box.id is not None else -1

            # Confidence
            conf = float(box.conf[0])

            # Class
            cls = int(box.cls[0])

            # Center point
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # Save data
            data.append({
                "frame": frame_idx,
                "track_id": track_id,
                "class": cls,
                "confidence": conf,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "center_x": cx,
                "center_y": cy
            })

            # Draw bounding box
            color = (0, 255, 0)
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)

            # Draw ID
            cv2.putText(
                frame,
                f"ID {track_id}",
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # Write frame
    out.write(frame)

    frame_idx += 1

# ---------------------------
# CLEAN UP
# ---------------------------
cap.release()
out.release()

# Save CSV
df = pd.DataFrame(data)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print("✅ Processing complete")
print(f"Video saved to: {OUTPUT_VIDEO_PATH}")
print(f"Tracking data saved to: {OUTPUT_CSV_PATH}")


0: 384x640 6 cars, 4.2ms
Speed: 0.9ms preprocess, 4.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 4.1ms
Speed: 0.8ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 4.1ms
Speed: 0.9ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 4.1ms
Speed: 0.9ms preprocess, 4.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)



0: 384x640 5 cars, 4.1ms
Speed: 0.9ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 4.1ms
Speed: 0.8ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 4.1ms
Speed: 0.9ms preprocess, 4.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 4.1ms
Speed: 0.8ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 4.1ms
Speed: 0.9ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 4.1ms
Speed: 0.8ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 truck, 4.1ms
Speed: 1.0ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 truck, 4.1ms
Speed: 0.8ms preprocess, 4.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384

In [ ]:
import cv2
import numpy as np
import pandas as pd

# ---------------------------
# CONFIG
# ---------------------------
VIDEO_PATH = "sim_dataset/videos/head-on/Town06_head-on_clear_01.mp4"
OUTPUT_VIDEO_PATH = "frame_differencing.mp4"
OUTPUT_CSV_PATH = "motion_data.csv"

MIN_CONTOUR_AREA = 500      # filter noise
MOTION_THRESHOLD = 2     # multiplier for spike detection

# ---------------------------
# VIDEO SETUP
# ---------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# ---------------------------
# INIT
# ---------------------------
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

motion_history = []
data = []

frame_idx = 1

# ---------------------------
# PROCESS VIDEO
# ---------------------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Frame differencing
    diff = cv2.absdiff(prev_gray, gray)

    # Blur + threshold
    blur = cv2.GaussianBlur(diff, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 25, 255, cv2.THRESH_BINARY)

    # Dilate to fill gaps
    dilated = cv2.dilate(thresh, None, iterations=2)

    # Find contours (motion regions)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    motion_area = 0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA:
            continue

        motion_area += area

        x, y, w, h = cv2.boundingRect(contour)

        # Draw bounding box
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

    motion_history.append(motion_area)

    # Compute motion spike (crash indicator)
    crash_flag = 0
    if len(motion_history) > 10:
        avg_motion = np.mean(motion_history[-10:])
        if avg_motion > 0 and motion_area > MOTION_THRESHOLD * avg_motion:
            crash_flag = 1
            cv2.putText(frame, "CRASH DETECTED", (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    # Save data
    data.append({
        "frame": frame_idx,
        "time_sec": frame_idx / fps,
        "motion_area": motion_area,
        "crash_flag": crash_flag
    })

    # Write frame
    out.write(frame)

    # Update
    prev_gray = gray
    frame_idx += 1

# ---------------------------
# CLEAN UP
# ---------------------------
cap.release()
out.release()

# Save CSV
df = pd.DataFrame(data)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print("✅ Done")
print(f"Saved video: {OUTPUT_VIDEO_PATH}")
print(f"Saved data: {OUTPUT_CSV_PATH}")

✅ Done
Saved video: frame_differencing.mp4
Saved data: motion_data.csv


In [ ]:
import json

with open("sim_dataset/video_annotations/Town03_head-on_clear_00.json/Town03_head-on_clear_00.json") as json_data:
    d = json.load(json_data)
    print(d)

{'base': [{'iteration': 51, 'objects': [{'id': 359, 'tag': 14, 'location': {'x': 5.86058922635857e-05, 'y': 0.0017932634800672531, 'z': 0.7014925479888916}, 'extent': {'x': 2.6787397861480713, 'y': 1.0166014432907104, 'z': 0.7053293585777283}, 'rotation': {'pitch': 0.0, 'yaw': 0.0, 'roll': 0.0}, '2d_bbox': [[1557, 75], [1717, 139]]}, {'id': 272, 'tag': 14, 'location': {'x': 0.023638123646378517, 'y': 0.0005998468259349465, 'z': 0.7926707863807678}, 'extent': {'x': 2.487122058868408, 'y': 1.0192005634307861, 'z': 0.7710590958595276}, 'rotation': {'pitch': 0.0, 'yaw': 0.0, 'roll': 0.0}, '2d_bbox': [[1376, 170], [1529, 239]]}, {'id': 207, 'tag': 14, 'location': {'x': -0.0003722693072631955, 'y': 6.978213491493079e-07, 'z': 0.6926480531692505}, 'extent': {'x': 2.0906050205230713, 'y': 0.9970585703849792, 'z': 0.6926480531692505}, 'rotation': {'pitch': 0.0, 'yaw': 0.0, 'roll': 0.0}, '2d_bbox': [[1339, 354], [1472, 425]]}, {'id': 151, 'tag': 14, 'location': {'x': -0.005347770173102617, 'y': 

In [21]:
import pandas as pd

labels = pd.read_csv("sim_dataset/labels.csv")
labels.set_index("rgb_path", inplace=True)
labels.head()

,annotations_path,type,accident_time,accident_frame,center_x,center_y,x1,y1,x2,y2,map,weather,camera_position,no_frames,duration,height,width,annotations_start_offset
rgb_path,,,,,,,,,,,,,,,,,,
videos/sideswipe/Town05_sideswipe_rain_44.mp4,video_annotations/Town05_sideswipe_rain_44.jso...,sideswipe,9.55,191,0.549219,0.387037,0.514583,0.350000,0.583854,0.424074,Town05,rain,44,391,19.55,1080,1920,31
videos/sideswipe/Town05_sideswipe_clear_00.mp4,video_annotations/Town05_sideswipe_clear_00.js...,sideswipe,8.65,173,0.494010,0.679167,0.453125,0.595370,0.534896,0.762963,Town05,clear,0,416,20.80,1080,1920,50
videos/sideswipe/Town05_sideswipe_sunset_03.mp4,video_annotations/Town05_sideswipe_sunset_03.j...,sideswipe,10.00,200,0.569531,0.890278,0.474479,0.781481,0.664583,0.999074,Town05,sunset,3,407,20.35,1080,1920,40
videos/sideswipe/Town05_sideswipe_night_30.mp4,video_annotations/Town05_sideswipe_night_30.js...,sideswipe,7.75,155,0.427604,0.600000,0.393229,0.551852,0.461979,0.648148,Town05,night,30,488,24.40,1080,1920,68
videos/sideswipe/Town05_sideswipe_clear_26.mp4,video_annotations/Town05_sideswipe_clear_26.js...,sideswipe,9.40,188,0.579948,0.261111,0.541146,0.195370,0.618750,0.326852,Town05,clear,26,468,23.40,1080,1920,33


In [29]:

print(labels.loc["videos/head-on/Town03_head-on_clear_00.mp4"])

annotations_path            video_annotations/Town03_head-on_clear_00.json.gz
type                                                                  head-on
accident_time                                                            3.45
accident_frame                                                             69
center_x                                                              0.65599
center_y                                                             0.151852
x1                                                                   0.588542
y1                                                                   0.087963
x2                                                                   0.723437
y2                                                                   0.215741
map                                                                    Town03
weather                                                                 clear
camera_position                                                 

In [39]:
import cv2
import numpy as np
import pandas as pd
from collections import deque

# ---------------------------
# CONFIG
# ---------------------------
# VIDEO_PATH = "input.mp4"
VIDEO_PATH = "sim_dataset/videos/head-on/Town06_head-on_clear_01.mp4"
OUTPUT_VIDEO_PATH = "output_tracked_collisions.mp4"
OUTPUT_CSV_PATH = "tracking_collision_data.csv"

MIN_CONTOUR_AREA = 500      # minimum blob to track
MAX_DISTANCE = 50           # max centroid distance to match across frames
CRASH_DISTANCE = 30         # distance threshold for collision
MOTION_HISTORY_FRAMES = 5   # frames to compute velocity drop

# ---------------------------
# VIDEO SETUP
# ---------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# ---------------------------
# INIT
# ---------------------------
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

next_id = 0
tracked_objects = {}  # id -> deque of centroids
frame_idx = 1
data = []

# ---------------------------
# HELPER FUNCTIONS
# ---------------------------
def compute_centroids(contours):
    centroids = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < MIN_CONTOUR_AREA:
            continue
        x, y, w, h = cv2.boundingRect(c)
        cx = x + w // 2
        cy = y + h // 2
        centroids.append((cx, cy))
    return centroids

def match_centroids(prev_tracked, new_centroids, next_id):
    """
    Simple nearest-neighbor matching
    Returns updated tracked_objects dict and updated next_id
    """
    updated = {}
    assigned = set()
    for tid, history in prev_tracked.items():
        last_pos = history[-1]
        # find nearest new centroid
        min_dist = float('inf')
        match = None
        for i, nc in enumerate(new_centroids):
            if i in assigned:
                continue
            dist = np.linalg.norm(np.array(last_pos) - np.array(nc))
            if dist < min_dist:
                min_dist = dist
                match = i
        if match is not None and min_dist < MAX_DISTANCE:
            # update history
            updated[tid] = history
            updated[tid].append(new_centroids[match])
            assigned.add(match)
    # assign new IDs to unmatched centroids
    for i, nc in enumerate(new_centroids):
        if i not in assigned:
            updated[next_id] = deque([nc], maxlen=10)
            next_id += 1
    return updated, next_id

def detect_collisions(tracked_objects):
    """
    Return list of pairs of ids that are colliding
    """
    collisions = []
    ids = list(tracked_objects.keys())
    for i in range(len(ids)):
        for j in range(i+1, len(ids)):
            id1, id2 = ids[i], ids[j]
            pos1, pos2 = tracked_objects[id1][-1], tracked_objects[id2][-1]
            dist = np.linalg.norm(np.array(pos1) - np.array(pos2))
            if dist < CRASH_DISTANCE:
                collisions.append((id1, id2))
    return collisions

# ---------------------------
# PROCESS VIDEO
# ---------------------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(prev_gray, gray)
    blur = cv2.GaussianBlur(diff, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 25, 255, cv2.THRESH_BINARY)
    dilated = cv2.dilate(thresh, None, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Compute centroids
    centroids = compute_centroids(contours)

    # Match centroids to existing tracked objects
    tracked_objects, next_id = match_centroids(tracked_objects, centroids, next_id)
    
    # Detect collisions
    collisions = detect_collisions(tracked_objects)

    # Draw centroids
    for tid, history in tracked_objects.items():
        cx, cy = history[-1]
        color = (0, 255, 0)
        cv2.circle(frame, (cx, cy), 5, color, -1)
        cv2.putText(frame, f"ID {tid}", (cx+5, cy-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Draw collisions
    for id1, id2 in collisions:
        c1 = tracked_objects[id1][-1]
        c2 = tracked_objects[id2][-1]
        cv2.line(frame, c1, c2, (0, 0, 255), 2)
        cv2.putText(frame, "CRASH", (min(c1[0],c2[0]), min(c1[1],c2[1])-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    # Save data for CSV
    for tid, history in tracked_objects.items():
        cx, cy = history[-1]
        collision_flag = 0
        for c in collisions:
            if tid in c:
                collision_flag = 1
                break
        data.append({
            "frame": frame_idx,
            "time_sec": frame_idx/fps,
            "track_id": tid,
            "center_x": cx,
            "center_y": cy,
            "collision_flag": collision_flag
        })

    out.write(frame)
    prev_gray = gray
    frame_idx += 1

# ---------------------------
# CLEAN UP
# ---------------------------
cap.release()
out.release()

df = pd.DataFrame(data)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print("✅ Done")
print(f"Video saved: {OUTPUT_VIDEO_PATH}")
print(f"Tracking and collisions saved: {OUTPUT_CSV_PATH}")

✅ Done
Video saved: output_tracked_collisions.mp4
Tracking and collisions saved: tracking_collision_data.csv
